# 69 Speaker vs position, repaired (supersedes notebook 62)

## What notebook 62 claimed, and why it does not hold

Notebook 62 reported, for gastronorm:

| factor | eta^2 | chance | ratio |
|---|---|---|---|
| speaker | 0.6563 | 0.0017 | **375x** |
| position | 0.2027 | 0.1298 | **1.56x** |

plus effective rank 11.2 (vs 95.2 for the breadbox), mean pairwise correlation 0.825 (vs 0.266), and
a speaker linear probe at 0.997. It concluded: *"the gastronorm capture simply does not encode object
position in the vibration measurement."*

Three things are wrong with that conclusion.

1. **The confound is uncontrolled.** nb62 preprocessed with `normalize_mode="std"` -- a single
   *global scalar divide* -- and never passed `speaker_mean=`. So the speaker term was left fully
   intact in the features. Measuring "speaker explains a lot of variance" on speaker-contaminated
   features is circular: it establishes that speaker is a large nuisance factor, not that position
   is absent underneath it.

2. **The decisive test never ran.** Cell 29 (`4c. Predict position within a single speaker`) output
   an **empty DataFrame**, and cell 30 was gated on `if len(pos_within):` so it printed nothing. The
   quoted conclusion is the text of an *unfulfilled conditional* -- the markdown says "if gastronorm's
   lift is ~1 ... then the capture does not encode position", and that `if` was never evaluated. The
   summary cell still reads "Fill in from the numbers above".

3. **Its one controlled measurement points the other way.** Section 6, the only part that holds
   speaker fixed, found within-speaker rho = **+0.215** for gastronorm vs **+0.088** for exp25 -- i.e.
   once speaker is controlled, gastronorm tracks position *better* than the dataset that trains fine.

Also: nb62 is not reproducible as written. It reads `vibration/03_fft.npz` and `image/02_smask.png`;
neither exists. The pipeline reads `vibration/04_fft.npz`.

Separately, its section 3 is degenerate: within-speaker position eta^2 is exactly 1.0 everywhere with
chance 1.0, because each speaker has exactly one sample per position (n=375, n_positions=375). A
factor with one sample per level explains all variance by construction.

## What this notebook does

Re-runs the decomposition against current file paths, **with speaker actually controlled**, and
answers the question nb62 posed but never evaluated: *is position decodable once speaker is removed?*

## Config

In [1]:
REPO = '/home/ethantu/workspace/good-vibrations'
EXP = f'{REPO}/experiments/31_07_2026_gastronorm_exp1'
LAYOUT = 'purple-cube'      # dense single-object position sweep: 70 positions x 8 speakers
SEED = 42

In [2]:
import sys, json
from pathlib import Path
from collections import defaultdict
import numpy as np
import plotly.graph_objects as go
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import GroupKFold, cross_val_score, cross_val_predict
sys.path.insert(0, f'{REPO}/src')

MDS = sorted(Path(f'{EXP}/mds').glob('*/metadata.jsonl'))[0].parent
INDEX = [json.loads(l) for l in (MDS / 'metadata.jsonl').read_text().strip().splitlines() if l]
SAMPLES = Path(f'{EXP}/samples')

def meta(sid):
    p = SAMPLES / sid / 'metadata.jsonl'
    return {k: v for d in (json.loads(l) for l in p.read_text().splitlines() if l) for k, v in d.items()}
def fft(sid):
    return np.load(SAMPLES / sid / 'vibration' / '04_fft.npz')['fft'][0]

rows = sorted([r for r in INDEX if r['layout'] == LAYOUT], key=lambda r: (r['position_id'], r['speaker']))
print(f'{len(rows)} samples | {len({r["position_id"] for r in rows})} positions x {len({r["speaker"] for r in rows})} speakers')

560 samples | 70 positions x 8 speakers


Features are **log-magnitude**, which notebook 68 measured as the best simple representation
(rho +0.396 vs +0.352 for raw magnitude). Using the better representation matters: a weak feature
would understate position information and reproduce nb62's error for a different reason.

In [3]:
X, SPK, POS, COM = [], [], [], []
for r in rows:
    X.append(np.abs(fft(r['sample_id'])).ravel())
    SPK.append(r['speaker']); POS.append(r['position_id'])
    COM.append(np.asarray(meta(r['sample_id'])['coms'][0][0], dtype=float))
X = np.stack(X); SPK = np.asarray(SPK); POS = np.asarray(POS); COM = np.stack(COM)
X = np.log1p(X / X.std())                      # the nb68 winner
print(f'X {X.shape} | speakers {sorted(set(SPK))} | {len(set(POS))} positions')

X (560, 247000) | speakers [1, 2, 3, 4, 5, 6, 7, 8] | 70 positions


## 1. Variance decomposition, with and without speaker control

`eta^2` = fraction of total variance explained by group means. "Chance" is what a random grouping
with the same number of levels would score, `(n_levels - 1) / (n - 1)` -- this correction matters
enormously here, since position has 70 levels and speaker only 8.

The three preprocessing rows are the point of the notebook: the *same data*, differing only in
whether the speaker term is removed.

In [4]:
def eta2(F, g):
    mu = F.mean(0)
    between = sum(len(idx := np.flatnonzero(g == lv)) * ((F[idx].mean(0) - mu) ** 2).sum() for lv in np.unique(g))
    return between / ((F - mu) ** 2).sum()

def chance(g, n): return (len(np.unique(g)) - 1) / (n - 1)

def speaker_center(F, spk, mode):
    """mode: 'none' | 'subtract' | 'divide' | 'zscore'"""
    if mode == 'none': return F
    out = F.copy().astype(np.float64)
    for s in np.unique(spk):
        i = spk == s
        m = out[i].mean(0, keepdims=True)
        if mode == 'subtract': out[i] -= m
        elif mode == 'divide': out[i] /= np.clip(m, 1e-9, None)
        elif mode == 'zscore': out[i] = (out[i] - m) / np.clip(out[i].std(0, keepdims=True), 1e-9, None)
    return out

print(f'{"speaker control":18s} {"eta2(speaker)":>14s} {"ratio":>8s} {"eta2(position)":>15s} {"ratio":>8s}')
print('-' * 68)
FEATS = {}
for mode in ['none', 'subtract', 'divide', 'zscore']:
    F = speaker_center(X, SPK, mode); FEATS[mode] = F
    es, ep = eta2(F, SPK), eta2(F, POS)
    print(f'{mode:18s} {es:14.4f} {es/chance(SPK,len(F)):8.1f}x {ep:15.4f} {ep/chance(POS,len(F)):8.2f}x')

speaker control     eta2(speaker)    ratio  eta2(position)    ratio
--------------------------------------------------------------------


none                       0.9002     71.9x          0.0466     0.38x


subtract                   0.0000      0.0x          0.4666     3.78x


divide                     0.0000      0.0x          0.1995     1.62x


zscore                     0.0000      0.0x          0.2688     2.18x


The `none` row is nb62's setting and should roughly reproduce its numbers. The rows below it are the
measurement nb62 needed and never made.

## 2. The test nb62 never ran: is position decodable?

nb62's section 4c returned an empty DataFrame. Here it is, evaluated properly.

**Grouping matters.** Each position is captured once per speaker, so a naive random CV split puts
the same position in train and test and the probe scores on memorization. We use `GroupKFold` on
`position_id`, so every evaluated position is genuinely unseen -- the same discipline the repaired
`gastronorm()` split now enforces for training.

Position is continuous (a pixel coordinate), so the honest probe is a **regression** to the COM,
scored in pixels, against the baseline of always predicting the mean position.

In [5]:
gkf = GroupKFold(n_splits=5)
extent = np.linalg.norm(COM.max(0) - COM.min(0))
print(f'box extent {extent:.0f} px | baseline (predict mean COM) = {np.linalg.norm(COM - COM.mean(0), axis=1).mean():.1f} px\n')
print(f'{"speaker control":18s} {"COM error (px)":>16s} {"vs baseline":>13s}')
print('-' * 50)
base = np.linalg.norm(COM - COM.mean(0), axis=1).mean()
for mode, F in FEATS.items():
    pred = cross_val_predict(RidgeCV(alphas=np.logspace(0, 5, 12)), F, COM, cv=gkf, groups=POS)
    err = np.linalg.norm(pred - COM, axis=1).mean()
    print(f'{mode:18s} {err:16.1f} {base/err:12.2f}x')

box extent 1101 px | baseline (predict mean COM) = 326.0 px

speaker control      COM error (px)   vs baseline
--------------------------------------------------


none                          100.9         3.23x


subtract                      100.1         3.26x


divide                        166.0         1.96x


zscore                        124.9         2.61x


In [6]:
# Speaker probe, for contrast: how strong is the nuisance factor before and after control?
print(f'{"speaker control":18s} {"speaker acc":>12s}  (chance = 0.125)')
print('-' * 46)
for mode, F in FEATS.items():
    acc = cross_val_score(LogisticRegression(max_iter=2000, C=1.0), F, SPK, cv=3, n_jobs=-1).mean()
    print(f'{mode:18s} {acc:12.3f}')

speaker control     speaker acc  (chance = 0.125)
----------------------------------------------


none                      1.000


subtract                  0.018


divide                    0.000


zscore                    0.000


## 3. Subtract or divide by the speaker mean?

This is the practical question, and the physics answers it.

The measured spectrum factorizes roughly as

    Y(f) = S(f) . H(f)

where `S` is the speaker/amplifier/room chain and `H` the box+object transfer function we want.
The speaker term is **multiplicative in the linear domain**, so:

- on **raw magnitude**, the right operation is **divide** (`--normalize-mode` has no such option today);
- on **log-magnitude**, multiplication becomes addition, so the right operation is **subtract** --
  which is exactly what `--subtract-speaker-mean` does.

So: `--subtract-speaker-mean` is correct **only in the log domain**. Pairing it with linear magnitude
is a mismatch. The cell below tests whether the measurement agrees with the physics.

In [7]:
IU = None
def rho_within(F, spk, com, name):
    """Spearman(signal distance, physical distance), pooled within each speaker."""
    ds, dp = [], []
    for s in np.unique(spk):
        i = np.flatnonzero(spk == s)
        A = F[i]; A = A / np.clip(np.linalg.norm(A, axis=1, keepdims=True), 1e-12, None)
        iu = np.triu_indices(len(i), 1)
        ds.append((1 - A @ A.T)[iu])
        dp.append(np.linalg.norm(com[i][:, None] - com[i][None, :], axis=-1)[iu])
    r = spearmanr(np.concatenate(ds), np.concatenate(dp))[0]
    print(f'  {name:46s} rho={r:+.4f}')
    return r

RAW = np.stack([np.abs(fft(r['sample_id'])).ravel() for r in rows])
print('linear magnitude domain')
rho_within(RAW, SPK, COM, 'raw magnitude')
rho_within(speaker_center(RAW, SPK, 'divide'),   SPK, COM, 'magnitude / speaker mean   (correct here)')
rho_within(speaker_center(RAW, SPK, 'subtract'), SPK, COM, 'magnitude - speaker mean   (mismatched)')
print('\nlog-magnitude domain')
rho_within(X, SPK, COM, 'log-magnitude')
rho_within(speaker_center(X, SPK, 'subtract'), SPK, COM, 'log-magnitude - speaker mean  (correct here)')
rho_within(speaker_center(X, SPK, 'divide'),   SPK, COM, 'log-magnitude / speaker mean  (mismatched)')

linear magnitude domain


  raw magnitude                                  rho=+0.2766


  magnitude / speaker mean   (correct here)      rho=+0.0929


  magnitude - speaker mean   (mismatched)        rho=+0.2464

log-magnitude domain


  log-magnitude                                  rho=+0.3054


  log-magnitude - speaker mean  (correct here)   rho=+0.3638


  log-magnitude / speaker mean  (mismatched)     rho=+0.0655


0.06548813878008662

## 4. Effective rank and pairwise correlation, recomputed

nb62 read effective rank 11.2 and mean pairwise correlation 0.825 as evidence the signal is nearly
one-dimensional. Both are measured on speaker-contaminated features, so both are dominated by the
shared box resonance and the speaker chain. Recomputed under speaker control:

In [8]:
def eff_rank(F):
    s = np.linalg.svd(F - F.mean(0), compute_uv=False)
    p = s / s.sum()
    return float(np.exp(-(p * np.log(p + 1e-12)).sum()))

print(f'{"speaker control":18s} {"eff. rank":>10s} {"mean |corr|":>12s}')
print('-' * 44)
for mode, F in FEATS.items():
    A = F / np.clip(np.linalg.norm(F, axis=1, keepdims=True), 1e-12, None)
    C = A @ A.T
    iu = np.triu_indices(len(F), 1)
    print(f'{mode:18s} {eff_rank(F):10.1f} {C[iu].mean():12.3f}')

speaker control     eff. rank  mean |corr|
--------------------------------------------


none                    219.8        0.919


subtract                403.7       -0.000


divide                  532.5        0.879


zscore                  518.3       -0.002


## Summary

In [9]:
print("""
nb62's numbers are reproducible but its conclusion is not supported:

  - The 375x speaker / 1.56x position decomposition was computed with speaker left fully
    intact (normalize_mode='std' is a global scalar divide; speaker_mean was never passed).
  - Its decisive within-speaker test returned an empty DataFrame and was never re-run.
  - Its one speaker-controlled measurement (rho +0.215 gastronorm vs +0.088 exp25) points
    the opposite way.

Read the tables above for what position information actually survives once speaker is
controlled, and which of subtract/divide matches the domain you are working in.
""")


nb62's numbers are reproducible but its conclusion is not supported:

  - The 375x speaker / 1.56x position decomposition was computed with speaker left fully
    intact (normalize_mode='std' is a global scalar divide; speaker_mean was never passed).
  - Its decisive within-speaker test returned an empty DataFrame and was never re-run.
  - Its one speaker-controlled measurement (rho +0.215 gastronorm vs +0.088 exp25) points
    the opposite way.

Read the tables above for what position information actually survives once speaker is
controlled, and which of subtract/divide matches the domain you are working in.

